# DTM fetch / merge / compressor

Scratch notebook for pulling BEV ALS DTM tiles (via `fetch_bev_als.py`), merging a
neighboring block of them into a single GeoTIFF, and (later) compressing the result.

## Fetch a 2x2 block of neighboring tiles

In [1]:
from fetch_bev_als import download_tiles

PRODUCT = "DTM"
OUT_DIR = "data/bev_als/DTM"

# 2x2 block of adjacent 50x50km tiles (same E columns, consecutive N rows).
TILE_IDS = [
    "N2650000E4400000", "N2650000E4450000",
    "N2700000E4400000", "N2700000E4450000",
]

tile_paths = download_tiles(PRODUCT, TILE_IDS, OUT_DIR)
tile_paths

N2650000E4400000: https://data.bev.gv.at/download/ALS/DTM/20250915/ALS_DTM_CRS3035RES50000mN2650000E4400000.tif -> data/bev_als/DTM\ALS_DTM_CRS3035RES50000mN2650000E4400000.tif


KeyboardInterrupt: 

## Merge the block into a single GeoTIFF

In [ ]:
import os

import rasterio
from rasterio.merge import merge

MERGED_PATH = os.path.join(OUT_DIR, f"merged_{PRODUCT}.tif")

srcs = [rasterio.open(p) for p in tile_paths]
try:
    mosaic, transform = merge(srcs)
    meta = srcs[0].meta.copy()
    meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": transform,
    })
    with rasterio.open(MERGED_PATH, "w", **meta) as dst:
        dst.write(mosaic)
finally:
    for src in srcs:
        src.close()

print(f"merged {len(tile_paths)} tiles -> {MERGED_PATH}")